# 第20讲：从真实轨迹优化 Skill
先核对同一个 `skill_body` 怎样进入优化、执行和导出，再看分数。默认单元不调用模型；最后一格必须经批准后主动启用。请使用装有 `gepa==0.0.27` 的内核，并在本目录打开。

In [ ]:
from pathlib import Path
import sys, subprocess
assert Path('adapter.py').is_file(), '请在第20讲目录打开'
subprocess.run([sys.executable, '-B', '-m', 'unittest', '-v', 'test_offline', 'test_enterprise'], check=True)


## 训练、验证与留出各做什么
训练轨迹用于反思；验证题用于比较和 Pareto 选择；留出题只有在候选选定并导出后才运行。下面仅显示划分，不执行模型。构造数据不能代表生产效果。

In [ ]:
from dataset import SPLITS, validate_splits
validate_splits()
{name: [(t['id'], t['expected']['cluster']) for t in tasks] for name, tasks in SPLITS.items()}


## 直接读两个官方适配方法
检查 `evaluate` 是否真的把候选正文传给执行器；检查反思记录是否保留错集群的调用参数和回执，而不只是一个低分。下一格展示本目录正在运行的实现，不展示另一份示意代码。

In [ ]:
import inspect
from adapter import OncallAdapter
from optimize import search, select_candidate
print(inspect.getsource(OncallAdapter))
print(inspect.getsource(search))
print(inspect.getsource(select_candidate))


## 接续第19讲
完成第19讲以后，按 README 为它的评测包准备三组题号清单。使用原执行器、快照和评分器，不重新抄写答案。默认的一道 ready 题不足以划成三组，不能复制凑数。下面两个路径留空时使用九题教学示例；填好以后使用第19讲的包。

In [ ]:
DATASET_MANIFEST = None  # 例如 Path('runtime/enterprise-splits.json')
SEED_SKILL = None  # 配合上项指定第19讲的 skills/oncall-investigation/SKILL.md
if DATASET_MANIFEST is not None:
    assert SEED_SKILL is not None, '请指定与第19讲工具配套的 Skill'
    subprocess.run([sys.executable, '-B', 'optimize.py', '--probe', '--dataset-manifest', str(DATASET_MANIFEST), '--skill', str(SEED_SKILL)], check=True)


## 获准后运行一次
按 README 在启动 Notebook 前设置 `GLM_API_KEY`、`GLM_MODEL`、`GLM_BASE_URL`。模型执行和反思必须共用已批准端点；API 费用由外部代理限制。先决定预算，再将下一格的开关改为 True。日志中的无改进或失败都应保留，不强制找到成功候选。中断后不要继续使用不完整结果。

In [ ]:
ALLOW_PAID_RUN = False
if ALLOW_PAID_RUN:
    command = [sys.executable, '-B', 'optimize.py', '--live', '--iterations', '1', '--minibatch', '1', '--metric-calls', '8', '--sessions', '14', '--reflections', '1', '--turns', '16']
    if DATASET_MANIFEST is not None:
        command += ['--dataset-manifest', str(DATASET_MANIFEST), '--skill', str(SEED_SKILL)]
    subprocess.run(command, check=True)
else:
    print('未启动任何模型调用。')


## 课后判断
阅读 README 的四道设计练习。真实运行后先读 REPORT.md，再逐题检查改善和退步、调用依据与成本。若导出仍是原版，先看是否没有严格改进，再排查优化组件、实际加载内容和导出变量是否一致。